# Week 06 - Synthetic Option Dataset

**Project:** AI/ML-Augmented Black-Scholes Framework
**Mandatory output of master plan section 11.6.**

The milestone for this notebook is one sentence from the plan:

> The student must demonstrate complete traceability
> **input parameters -> BS quantities -> features -> target.**

Everything here imports from `ml_dataset.py`, which carries its own self-test.
Nothing is re-implemented in the notebook, so the notebook and
`postmidsem_week6.py` cannot drift apart.


In [ ]:
import sys, os

# Run from the repository root: the modules live there, and every path in the
# project (data/, results/, ml_plots/) is relative to it. Without the chdir a
# notebook launched from notebooks/ would quietly write a second copy of the
# dataset into notebooks/data/.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working directory:", os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (11, 3.8), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

from ml_dataset import (SEED, build_dataset, save_dataset, load_dataset,
                        verify_traceability, DATA_PATH,
                        FEATURES_BASIC, FEATURES_EXTENDED, TARGET)
from black_scholes import bs_price

print("seed", SEED)
print("basic features   ", FEATURES_BASIC)
print("extended features", FEATURES_EXTENDED)
print("target           ", TARGET, " = C_BS / K")

## 1. The sampling design

Four choices here are deliberate and are argued in full in
`ML_experiment_plan.md` section 6. In brief:

| Choice | Reason |
|---|---|
| `S` fixed at 100 | Black-Scholes is homogeneous of degree one in `(S, K)`, so `C/K` depends on them only through `m = ln(S/K)`. Sampling `S` as well would be redundant. |
| `K` log-uniform | makes `m` **uniform**, so moneyness is covered evenly rather than bunching at the money. |
| `T` log-uniform | short maturities are where the surface is most curved. Uniform `T` on `[0.02, 2]` would leave only about 1% of rows below `T = 0.04`. |
| calls only | a call and a put at the same parameters share a feature vector but not a price. Puts follow exactly by parity, so they add no information and would make the target two-valued. |

The last one was decided by measurement, not assumption - the cost is
quantified in the regression notebook.

In [ ]:
df = build_dataset(n=20000, seed=SEED, verbose=True)
df.head()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].hist(df["m"], bins=40, color="#4878a8", edgecolor="white")
ax[0].set_xlabel("m = ln(S/K)"); ax[0].set_title("uniform by construction")
ax[1].hist(np.log10(df["T"]), bins=40, color="#4878a8", edgecolor="white")
ax[1].set_xlabel(r"$\log_{10} T$"); ax[1].set_title("log-uniform in T")
s = df.sample(4000, random_state=SEED)
sc = ax[2].scatter(s["m"], s["y"], c=np.log10(s["T"]), s=4, cmap="viridis", alpha=0.6)
ax[2].set_xlabel("m"); ax[2].set_ylabel("y = C/K")
ax[2].set_title("the surface to be learned")
plt.colorbar(sc, ax=ax[2], label=r"$\log_{10} T$")
plt.tight_layout(); plt.show()

print("%.1f%% of rows have T < 0.04" % (100 * (df['T'] < 0.04).mean()))

The right-hand panel is the whole problem in one picture: flat for
`m < 0`, rising like `e^m` deep in the money. That convexity is what no linear
model can reproduce, and it is why the regression notebook predicts the sign
of its own residuals before fitting anything.

## 2. The split

70 / 15 / 15, drawn **once** under the seed and written into the file as a
column. Downstream notebooks read it and never re-draw it, which is what
guarantees the Week 7 models are scored on exactly this test set.

Random splitting is legitimate here and only here, because synthetic rows are
independent draws with no time index. For real option data the split becomes
chronological (master plan 5.11); that change is recorded in
`ML_experiment_plan.md` section 3.3.

In [ ]:
print(df["split"].value_counts(normalize=True).round(4).to_string())
print()
print("disjoint by construction: every row carries exactly one label")
print("labels present:", sorted(df["split"].unique()))
print("unlabelled rows:", int(df["split"].isna().sum()))

## 3. Save, then audit from disk

The audit deliberately **reloads the CSV** rather than using the DataFrame in
memory. That also tests the round-trip: if the file were written with rounded
floats, the recomputation would no longer match and the chain would break.

In [ ]:
save_dataset(df, DATA_PATH)
disk = load_dataset(DATA_PATH)
print("reloaded", len(disk), "rows,", disk.shape[1], "columns\n")
tr = verify_traceability(disk, verbose=True)

### The chain on a single row

The table above is the aggregate. Here is one row end to end, so the chain is
readable by eye rather than only by assertion.

In [ ]:
i = int(np.argmin(np.abs(disk["m"].to_numpy())))
row = disk.iloc[i]

print("1. INPUTS")
print("   S=%.4f  K=%.4f  T=%.6f  r=%.4f  sigma=%.4f  q=%.4f  %s"
      % (row.S, row.K, row['T'], row.r, row.sigma, row.q, row.option_type))

rec = float(bs_price(row.S, row.K, row['T'], row.r, row.sigma, row.q,
                     option=row.option_type))
print("\n2. BLACK-SCHOLES QUANTITIES")
print("   stored C_BS = %.12f" % row.C_BS)
print("   recomputed  = %.12f     difference %.2e" % (rec, abs(rec - row.C_BS)))
print("   Delta=%.6f  Gamma=%.6f  Vega=%.6f" % (row.delta, row.gamma, row.vega))

print("\n3. FEATURES")
print("   m = ln(S/K) = ln(%.4f / %.4f) = %.12f" % (row.S, row.K, np.log(row.S/row.K)))
print("   stored m                      = %.12f" % row.m)

print("\n4. TARGET")
print("   y = C_BS / K = %.6f / %.4f = %.12f" % (row.C_BS, row.K, row.C_BS/row.K))
print("   stored y                       = %.12f" % row.y)

print("\n5. SPLIT MEMBERSHIP:", row.split)

## 4. Independent structural check

Every check so far recomputes with the same pricer that generated the data, so
agreement is necessary but not sufficient - a bug inside `bs_price` would be
invisible to it.

Put-call parity is independent of that. It is an arbitrage identity, not a
property of the formula, and it must hold for the sampled parameters whatever
the pricer does:

$$C - P = S e^{-qT} - K e^{-rT}$$

In [ ]:
from black_scholes import put_call_parity_error
err = put_call_parity_error(disk["S"].to_numpy(), disk["K"].to_numpy(),
                            disk["T"].to_numpy(), disk["r"].to_numpy(),
                            disk["sigma"].to_numpy(), disk["q"].to_numpy())
print("max |parity error| over %d parameter sets: %.3e" % (len(disk), np.abs(err).max()))
print("\nThis holds even though the file contains no put rows: the check is")
print("computed from the parameters, not from stored put prices.")

## 5. Milestone

| Stage | Verified |
|---|---|
| inputs -> BS quantities | recomputed from the stored columns |
| BS quantities -> features | `m = ln(S/K)` reproduced exactly |
| features -> target | `y = C_BS / K` reproduced exactly |
| independent check | put-call parity on the sampled parameters |
| split | stored in the file, disjoint, 70/15/15 |

Run `python ml_dataset.py` from the repository root to execute the full
self-test (34 checks) on the module itself.